In [ ]:
# 에이전트 및 그래프 프레임워크 설치
!pip install -U langgraph langchain langchain-openai langchain-community

# 검색 및 외부 도구 API (세미나 자료 근거 )
!pip install -U tavily-python google-search-results

# 데이터 처리 및 벡터 DB (메모리 구현용 )
!pip install chromadb openai praw

# 비동기 및 유틸리티
!pip install -U nest_asyncio tqdm

In [ ]:
# ==========================================
# 1. Standard Library & Basic Imports
# ==========================================
import os
import re
import json
import asyncio
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
import praw
import ast
import chromadb
import random
import nest_asyncio
import operator

# ==========================================
# 2. From Imports (Specific Modules)
# ==========================================

# 데이터 타이핑
from typing import Annotated, List, TypedDict, Union, Optional

# 기초 및 유틸리티 (Basic & Utilities)
from datetime import datetime
from tqdm.auto import tqdm  # tqdm과 중복되나 auto 권장
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# AI 및 API 클라이언트 (AI & API Clients)
from openai import AsyncOpenAI

# 벡터 DB 및 임베딩 (Vector DB & Embeddings)
import chromadb
from chromadb.utils import embedding_functions
from langchain_community.vectorstores import Chroma

# LangChain - LLM 프레임워크 (LangChain Framework)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode

nest_asyncio.apply() # 비동기 루프 허용

In [ ]:
# ==========================================
# 3. Environment Setting
# ==========================================

# Reddit
client_id = os.getenv("CLIENT_ID")
client_secret = os.getenv("CLIENT_SECRET")
user_agent = os.getenv("USER_AGENT")

reddit = praw.Reddit(
    client_id = client_id,
    client_secret = client_secret,
    user_agent = user_agent
)

# llm 모델
llm = ChatOpenAI(
    model = "gpt-5.1",
    model_kwargs = {"service_tier" : "flex"},
    temperature = 0,
    api_key = os.getenv("OPENAI_API_KEY"),
    response_format = {"type" : "json_object"}
)

# 임베딩 모델
embeddings = OpenAIEmbeddings()

# 벡터 DB
vector_db = Chroma(
    collection_name = "research_cr_memory",
    embedding_functions = embeddings,
    persist_directory = "./chroma_db"
)

In [ ]:
# ==========================================
# 4. Class and Function Definitions
# ==========================================

# GraphState Class
class GraphState(TypedDict):
    # Raw Data
    raw_review : str

    # Agent 판단 결과
    is_valid : bool
    purposes : List[str]
    reasoning : str

    # 중간 결과물 및 메모리
    segments : List[str]
    cr_inventory : Annotated[List[str], operator.add]

    # Customer Requirement
    extracted_crs : List[dict]

# LLM-Tagging 함수 정의
def get_openai_tags(llm, review_text):
    # 시스템 프롬프트: 역할과 출력 형식을 강력하게 정의
    system_instruction = """
        ### Role: Senior NPD Research Data Auditor

        Your sole mission is to determine if a review is "Actionable Product Feedback"
        and classify its usage purpose.

        ## Step 1. Validity Check (is_valid):
        Do NOT look for noise patterns. Instead, ask yourself:

        "Can I complete this sentence from this review?
        'A product designer should [do X] because this user needs [Y].'"

        - If YES → proceed to Step 2
        - If NO  → is_valid: false, primary_purpose: "None", stop here

        A valid review must satisfy ALL of the following:
        1. It is a self-contained statement (not a reply fragment or conversation).
        2. It expresses a concrete preference, expectation, or complaint about
        the product's design, performance, or ergonomics.
        3. A product designer can extract a concrete requirement from it.
        The primary intent is NOT to seek help, ask for recommendations,
        or get answers from other users.

        ## Step 2. Evidence Collection (BEFORE assigning purpose):
        List the specific signals in the review that indicate Professional or Casual intent.

        - You MUST find at least 2 concrete, distinct signals to assign a purpose.
        - If you cannot find 2 signals, set primary_purpose: "None", is_valid: false.

        Signals for **Professional**:
            The laptop is used as a tool for specialized, skill-based work
            where performance directly impacts output quality or productivity.
            (e.g., software dev, data science, 4K editing, 3D rendering,
            heavy corporate multitasking, engineering tools, etc.)

        Signals for **Casual**:
            The laptop is used primarily for consumption, communication,
            or light tasks without domain-specific performance demands.
            (e.g., web browsing, streaming, social media, 
            basic student homework, everyday home use, etc.)

        ## Step 3. Confidence Check:
        Ask yourself: "If I read this review 10 times, would I assign 
        the same purpose every time with >90% certainty?"

        - If YES → assign the purpose
        - If NO, or if Professional and Casual signals are mixed/equal → 
        primary_purpose: "None", is_valid: false

        ## Output Schema (JSON Only):
        Important: Fill in "purpose_signals" BEFORE deciding "primary_purpose".
        {
            "purpose_signals": ["signal 1 from the review", "signal 2 from the review"],
            "is_valid": boolean,
            "primary_purpose": "Professional" | "Casual" | "None",
            "designer_action": "Complete the sentence: 'A designer should [X] because this user needs [Y].' Leave empty string if is_valid is false.",
            "reasoning": "One sentence explaining the validity and purpose decision.",
            "evidence_snippet": "The most relevant quote. Empty string if invalid."
        }
    """

    user_prompt = f"""
        Review: "{review_text}"""
    
    try:
        message = [
            SystemMessage(content = system_instruction),
            HumanMessage(content = user_prompt)
        ]
        response = llm.invoke(message)
        
        return response.content
    except Exception as e:
        print(f"Error : {e}")
        return None

# 파싱
def parse_purpose(text): # purpose_classification 컬럼의 값을 파싱하여 리스트로 반환
    try:
        return ast.literal_eval(text)
    except (ValueError, SyntaxError):
        return re.findall(r"'([^']*)'", text)
    
# CR Extraction 함수 정의
# 목적별 Customer Requirement 추출용 프롬프트 함수
def get_cr_extraction_prompt(purpose, existing_knowledge, designer_action):
    return f"""
        ### Identity: The Voice of the "{purpose}" User Group
        You are the representative of the "{purpose}" community. You understand their lifestyle, daily challenges, and the specific performance standards they demand from a laptop. 

        ### Reference Knowledge (Existing Memory):
        The following terms are already standardized in our database. If the current review's needs match any of these semantically, you MUST use the exact same term to ensure data consistency:
        {existing_knowledge if existing_knowledge else "No existing terms yet."}

        ### Design Guidance from Auditor:
        Focus your extraction on the following direction provided by the Tagger agent:
        "{designer_action}"

        ### Mission:
        Read the following review segments and extract **Atomic Customer Requirements (CR)**. 
        A "CR" must represent a single, distinct need of the user. Your goal is to provide concise, standardized labels that can be directly mapped to engineering characteristics in a QFD matrix.

        ### CR Extraction Rules (Strict):
        1. **Atomic & Single-Intent**: Each requirement must contain only ONE specific need. If a review mentions weight and battery, split them into two separate CRs. 
        2. **Noun-Phrase Format**: Use short, punchy noun phrases. Avoid full sentences.
        3. **Exclude Reasons & Context**: Do not include "why" or "how" in the requirement field. Put all supporting context in the 'evidence_summary'. 
        4. **Objective Translation**: Convert subjective feedback into objective standards where possible.
        5. **Term Consistency (CRITICAL)**: Prioritize mapping to 'Reference Knowledge' before creating a new term. If you create a new term, ensure it follows the naming style of existing ones.

        ### Examples:
        - ❌ Bad: "Lightweight chassis under 1.5kg for better portability during commute."
        - ✅ Good: "Body weight (under 1.5kg)"
        - ❌ Bad: "High-resolution screen with good color for photo editing."
        - ✅ Good: "Color accuracy (DCI-P3 100%)"

        ### Output Schema (JSON only):
        {{
            "representative_persona": "Briefly describe your identity as a {purpose} user",
            "requirements": [
                {{
                    "requirement": "The atomic, noun-phrase label (e.g., 'Battery life', 'Fan noise')",
                    "is_mapped_from_memory": "boolean (true if you used a term from Reference Knowledge, false if new)",
                    "evidence_summary": "Short snippet of what users actually said"
                }}
            ]
        }}
    """

# Customer Requirement 추출 함수 정의
def extraction_node(state: GraphState):
    # 1. 장부(State)에서 데이터 가져오기
    purpose = state["primary_purpose"]
    review_text = state["raw_review"]
    designer_action = state["designer_action"]
    
    print(f"--- [에이전트: Extractor] {purpose} 관점에서 분석 시작 ---")

    # 2. Chroma DB에서 기존 지식 검색 (RAG)
    try:
        # 데이터가 하나도 없는 초기 상태에서도 오류 없이 빈 결과를 반환합니다.
        related_docs = vector_db.similarity_search(review_text, k=5)
        existing_knowledge = "\n".join([f"- {doc.page_content}" for doc in related_docs])
    except Exception as e:
        print(f"Chroma DB 검색 중 알림: {e} (초기 단계일 수 있음)")
        existing_knowledge = "No existing terms yet."

    # 3. 프롬프트 생성 (existing_knowledge 전달)
    system_prompt = get_cr_extraction_prompt(purpose, existing_knowledge, designer_action)
    user_prompt = f"Target Review to Analyze: \n{review_text}"

    # 4. LangChain llm.invoke 호출 (client 방식 제거)
    try:
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        
        # llm은 gpt-5.1-flex로 설정된 LangChain 객체입니다.
        response = llm.invoke(messages)
        
        # JSON 파싱 (마크다운 태그 등 노이즈 제거)
        content = response.content.replace("```json", "").replace("```", "").strip()
        result = json.loads(content)
        
        # 5. 새로운 지식(CR)을 Chroma DB에 저장 (지식 축적)
        new_requirements = result.get('requirements', [])
        new_terms_to_save = [
            req['requirement'] 
            for req in new_requirements 
            if not req.get('is_mapped_from_memory', True)
        ]
        
        if new_terms_to_save:
            vector_db.add_texts(new_terms_to_save)
            print(f">>> [Memory] {len(new_terms_to_save)}개의 신규 용어를 DB에 등록했습니다.")

        # 6. 장부(State) 업데이트 결과 반환
        return {
            "extracted_crs": new_requirements,
            "cr_inventory": [req['requirement'] for req in new_requirements]
        }

    except Exception as e:
        print(f"Extraction 노드 실행 중 오류 발생: {e}")
        return {"extracted_crs": []}